# Kernel cuantico ZZ para QSVM

Construye el kernel de fidelidad ``K(x_i, x_j) = P(00...0)`` del circuito
``U(x_j)^dagger U(x_i)``, con el feature map ZZ definido en pytket y ejecutado via guppy.

Flujo: cargar datos -> inspeccionar circuitos (sin shots) -> construir la matriz
kernel. Toda la logica vive en `funciones_nexus.py`; aqui solo quedan los
parametros y las llamadas.

La construccion de la matriz esta apagada por defecto (`RUN_MATRIX = False`):
revisa circuitos y costo antes de encenderla.

## 1. Configuracion y datos

Carga el dataset escalado y lo separa en train/test segun `_PartInd_`.

In [1]:
import pandas as pd
from IPython.display import display
from pytket.circuit.display import render_circuit_jupyter

# Los valores del kernel suelen ser pequenos; se muestran con 6 decimales
# fijos (sin notacion cientifica) para poder distinguirlos.
pd.set_option("display.float_format", "{:.6f}".format)

from funciones_nexus import (
    cargar_datos_kernel,
    zz_feature_map,
    seleccionar_par_kernel,
    iniciar_matriz_kernel,
    consultar_matriz_nexus,
    guardar_kernel_qsvm,
    MATRIX_BACKEND_OPTIONS,
)

PROJECT_NAME = "prueba_migracion"                       # Proyecto de Nexus (se crea si no existe)
KERNEL_DATA_PATH = "data/processed/df_escalado.csv"     # Dataset escalado con _PartInd_

kernel_df, kernel_feature_columns, kernel_train_df, kernel_test_df = cargar_datos_kernel(KERNEL_DATA_PATH)
print("Features:", kernel_feature_columns)
print(f"Train: {kernel_train_df.shape} | Test: {kernel_test_df.shape}")
display(kernel_train_df.head())

Features: ['ph', 'Hardness', 'Solids', 'Chloramines', 'Sulfate', 'Conductivity', 'Organic_carbon', 'Trihalomethanes', 'Turbidity']
Train: (2620, 9) | Test: (656, 9)


,ph,Hardness,Solids,Chloramines,Sulfate,Conductivity,Organic_carbon,Trihalomethanes,Turbidity
0,0.505288,0.371295,0.048040,-0.791809,2.052308,0.925183,0.565242,-0.365393,-0.177457
1,-0.709433,0.935084,-1.016080,0.850917,1.617777,-1.501693,-2.169448,-1.581404,-0.486502
2,0.929048,0.980710,0.808192,-0.585714,-0.019216,-0.818137,0.085404,-0.351573,0.547202
3,-0.227728,0.671248,-0.001853,0.030738,-0.019216,-0.677955,0.774861,0.501623,-0.091554
4,-0.390242,0.176673,-0.548156,-0.571875,-0.196177,0.665479,0.686438,-1.263826,-0.544746


## 2. Inspeccion del feature map U(x)

No consume shots.

In [2]:
PREVIEW_ROW = 0     # Cambia esta fila para inspeccionar otro U(x), sin ejecutar shots

preview_x = kernel_train_df.iloc[PREVIEW_ROW].to_numpy(dtype=float)
feature_map_preview = zz_feature_map(preview_x)
print(f"Feature map de train[{PREVIEW_ROW}] | qubits: {feature_map_preview.n_qubits} | puertas: {feature_map_preview.n_gates}")
render_circuit_jupyter(feature_map_preview)

Feature map de train[0] | qubits: 9 | puertas: 163


## 3. Seleccion e inspeccion del par

Construye ``U(x_j)^dagger U(x_i)`` con barreras para revisarlo antes de ejecutar.

In [3]:
KERNEL_ROW_I = 0    # Filas de train que forman el par
KERNEL_ROW_J = 1

kernel_x_i, kernel_x_j, kernel_preview_circuit = seleccionar_par_kernel(
    kernel_train_df, KERNEL_ROW_I, KERNEL_ROW_J
)
render_circuit_jupyter(kernel_preview_circuit)

Kernel seleccionado: train[0] vs train[1]
Qubits: 9
Puertas: 326
Profundidad: 220


## 4. Matriz kernel

Ejecuta el triangulo superior y refleja por simetria (`K(i,j) = K(j,i)`).
Con la diagonal desactivada se fija `K(i,i) = 1` sin ejecutar circuitos:
para `m` filas se requieren `m(m-1)/2` circuitos.

Backends: Selene local o Nexus (Selene, H1/H2 via compile job de pytket, Helios).
Tras enviar a Nexus, **no reejecutes esta celda**: usa la celda de consulta.

In [ ]:
MATRIX_ROWS = [0, 1, 2, 3]                    # Filas de train que forman la matriz
MATRIX_BACKEND = "H2-EMULATOR"   # Ver MATRIX_BACKEND_OPTIONS
RUN_MATRIX = True                          # Interruptor de seguridad
MATRIX_SHOTS = 10
MATRIX_SEED = 42
MATRIX_EXECUTE_DIAGONAL = True               # False: fija K(i,i)=1 sin ejecutar
SAVE_MATRIX_RUN = True

matrix_state, matrix_result = iniciar_matriz_kernel(
    kernel_train_df, MATRIX_ROWS, MATRIX_BACKEND, RUN_MATRIX,
    n_shots=MATRIX_SHOTS, seed=MATRIX_SEED,
    ejecutar_diagonal=MATRIX_EXECUTE_DIAGONAL,
    guardar=SAVE_MATRIX_RUN, project_name=PROJECT_NAME,
)

if matrix_result is not None:
    display(pd.DataFrame(matrix_result["kernel_matrix"], index=MATRIX_ROWS, columns=MATRIX_ROWS))
    display(matrix_result["run_summary"])

Already logged in. Tokens are valid.


Preparando pytket_circuit: 100%|██████████| 10/10 [00:04<00:00,  2.27circuito/s]


Circuitos pytket subidos; compile job enviado.
Compile Job ID: 4f406f29-fd2f-4ea6-8a86-e10b5b4c9784
Backend: H2-Emulator
Formato: pytket_circuit
Programas: 10
Consulta el avance con la celda de consulta; no reenvies esta celda.


## 5. Consulta de la matriz remota

Reejecutar solo esta celda. Para H1/H2 encadena compile -> execute automaticamente;
al completarse reconstruye la matriz y guarda el CSV.

In [12]:
matrix_state, matrix_result_remoto = consultar_matriz_nexus(matrix_state, guardar=SAVE_MATRIX_RUN)

if matrix_result_remoto is not None:
    display(pd.DataFrame(matrix_result_remoto["kernel_matrix"], index=MATRIX_ROWS, columns=MATRIX_ROWS))
    display(matrix_result_remoto["run_summary"])

Execute Job ID: 407bab2c-d427-41de-969e-8ed7c614bbf7
Execute status: JobStatusEnum.COMPLETED
Execute message: The job is completed.
Matriz kernel reconstruida.
Run de matriz guardado en: data\runs\kernel_matrix_run_407bab2c-d427-41de-969e-8ed7c614bbf7.csv


,0,1,2,3
0,0.9,0.0,0.0,0.0
1,0.0,0.9,0.0,0.0
2,0.0,0.0,0.8,0.0
3,0.0,0.0,0.0,0.9


,matrix_i,matrix_j,row_i,row_j,seed,result_id,backend,program_format,n_qubits,zero_state,zero_count,shots,kernel_rate
0,0,0,0,0,42,a6570821-ada7-4c7c-8e25-75be8ecdabf3,H2-Emulator,pytket_circuit,9,000000000,9,10,0.9
1,0,1,0,1,43,e57a7b8b-5146-4ed9-adaa-01136165bdfd,H2-Emulator,pytket_circuit,9,000000000,0,10,0.0
2,0,2,0,2,44,80a03a1b-f681-4ec7-adcc-31d21a3b9b0e,H2-Emulator,pytket_circuit,9,000000000,0,10,0.0
3,0,3,0,3,45,0cc7c65e-7223-49d5-b71f-c6a6779e1f1e,H2-Emulator,pytket_circuit,9,000000000,0,10,0.0
4,1,1,1,1,47,6c8aea67-3638-4bad-ac6e-04043cc66de1,H2-Emulator,pytket_circuit,9,000000000,9,10,0.9
5,1,2,1,2,48,fefb4834-c96c-4f88-9bf0-0c77414baaf5,H2-Emulator,pytket_circuit,9,000000000,0,10,0.0
6,1,3,1,3,49,59be0e6d-32e9-496c-a5d7-c80e4499314b,H2-Emulator,pytket_circuit,9,000000000,0,10,0.0
7,2,2,2,2,52,a6c92a54-13ee-4e94-8b71-272100b3158f,H2-Emulator,pytket_circuit,9,000000000,8,10,0.8
8,2,3,2,3,53,b1d79750-42fc-4a95-b7c5-ce90ec702d32,H2-Emulator,pytket_circuit,9,000000000,0,10,0.0
9,3,3,3,3,57,08be5f78-0dbb-4c08-ab06-2269751054fa,H2-Emulator,pytket_circuit,9,000000000,9,10,0.9


## 6. Guardado del kernel para QSVM

Persiste la matriz de Gram **cuadrada** (lista para `SVC(kernel="precomputed")`) y un CSV de metadatos con su procedencia
(backend, job_id, shots, filas, timestamp). Toma la matriz local o la remota, la que este disponible.

In [13]:
# Guarda la matriz de Gram cuadrada + metadatos (data/runs/kernel_qsvm_*.csv).
if matrix_result_remoto is not None:
    resultado_final, fuente, id_job = matrix_result_remoto, f"nexus_{MATRIX_BACKEND}", matrix_state["job_ref"].id
elif matrix_result is not None:
    resultado_final, fuente, id_job = matrix_result, "local_statevector", None
else:
    resultado_final = None
    print("Aun no hay una matriz kernel construida que guardar. Corre el paso 4 (o 5).")

if resultado_final is not None:
    ruta_kernel, ruta_meta = guardar_kernel_qsvm(resultado_final, source=fuente, job_id=id_job)
    print("Matriz kernel guardada en:", ruta_kernel)
    print("Metadatos en:", ruta_meta)
    display(pd.read_csv(ruta_kernel, sep=";", index_col=0))

Matriz kernel guardada en: data\runs\kernel_qsvm_407bab2c-d427-41de-969e-8ed7c614bbf7.csv
Metadatos en: data\runs\kernel_qsvm_407bab2c-d427-41de-969e-8ed7c614bbf7_meta.csv


,0,1,2,3
0,0.9,0.0,0.0,0.0
1,0.0,0.9,0.0,0.0
2,0.0,0.0,0.8,0.0
3,0.0,0.0,0.0,0.9
